# Integrated TeleLens System

One system, four capabilities: shared embeddings, triage with deferral, digests, and question answering.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PACKAGES = ["rouge-score", "pytest", "sentencepiece", "protobuf"]
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    if PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    hits = glob.glob("/kaggle/input/**/telelens/__init__.py", recursive=True)
    if hits:
        ROOT = Path(hits[0]).parents[2]
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BharatChandra-sys/NLP-case-study.git", "/tmp/repo"], check=True)
        ROOT = Path("/tmp/repo")
    RESULTS = Path("/kaggle/working/results/integrated")
else:
    ROOT = Path.cwd().resolve().parent
    RESULTS = ROOT / "results" / "integrated"
sys.path.insert(0, str(ROOT / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def save_json(name, payload):
    (RESULTS / name).write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", str(ROOT / "tests"), "-q", "-p", "no:cacheprovider"], capture_output=True, text=True,
                        env={**os.environ, "PYTHONPATH": str(ROOT / "src")})
print(result.stdout[-1500:], result.stderr[-500:])
(RESULTS / "pytest.txt").write_text(result.stdout, encoding="utf-8")

In [ ]:
from telelens.data import load_comcast, load_telecom
from telelens.pipeline import TeleLens

telecom = load_telecom(10000)
comcast = load_comcast()
system = TeleLens(telecom, comcast, "deepset/roberta-base-squad2", device=DEVICE)
reviews = ["my broadband keeps disconnecting and customer service has not fixed it",
           "the engineer arrived on time and the router was set up in minutes",
           "it was fine i suppose though the price went up after a few months",
           "the speed is good but nobody answers when you call and the bill was wrong once",
           "great deal at first however the router needed replacing twice and they sorted it eventually"]
triage = [{"review": r, **system.triage(r)} for r in reviews]
virgin = telecom[[("virgin" in t.split()) for t in telecom["text"]]].sample(80, random_state=SEED)["text"].tolist()
digest = system.digest(virgin)
questions = ["What was the cancellation fee for the customer who moved to a neighborhood not serviced by Comcast?",
             "Which insurance company refused to pay the full electrician estimate after faulty cable wiring?",
             "How much does Comcast charge for 5G home internet in Seattle?"]
answers = [{"question": q, **system.ask(q)} for q in questions]
neighbours = {w: system.embeddings.neighbours(w, 5) for w in ["reliable", "not_reliable", "unstable"]}
save_json("demo.json", {"triage": triage, "digest": digest, "answers": answers, "neighbours": neighbours})
print(json.dumps({"triage": triage, "digest": digest, "answers": answers, "neighbours": neighbours}, indent=1, default=float)[:6000])

## Recorded Kaggle results

Executed on a Kaggle T4 GPU as `telelens-05-integrated`. Files are in `results/integrated/`.

```json
{
 "triage": [
  {
   "review": "my broadband keeps disconnecting and customer service has not fixed it",
   "prediction": "Bad",
   "probability_good": 0.03949878537255682,
   "decision": "automatic",
   "similar": [
    {
     "cosine": 0.49539025102608214,
     "label": "Bad",
     "text": "terrible keeps disconnecting"
    },
    {
     "cosine": 0.2306022342416904,
     "label": "Bad",
     "text": "terrible internet connection though out the house keeps disconnecting upstairs is the worst even with a plug in pod u supplied"
    },
    {
     "cosine": 0.20626352539065002,
     "label": "Bad",
     "text": "the service is weak in most parts of my house my bathroom kitchen and kids room it keeps disconnecting wifi calls"
    }
   ]
  },
  {
   "review": "the engineer arrived on time and the router was set up in minutes",
   "prediction": "Good",
   "probability_good": 0.939163652846009,
   "decision": "automatic",
   "similar": [
    {
     "cosine": 0.30822851294767084,
     "label": "Good",
     "text": "good service arrived on time"
    },
    {
     "cosine": 0.29167851710455966,
     "label": "Good",
     "text": "the engineer arrived on time and started work straight was friendly polite and went as expected and new wifi up and took about hours or so"
    },
    {
     "cosine": 0.24259079198287486,
     "label": "Good",
     "text": "engineer arrived next day good service"
    }
   ]
  },
  {
   "review": "it was fine i suppose though the price went up after a few months",
   "prediction": "Good",
   "probability_good": 0.8614325187934918,
   "decision": "automatic",
   "similar": [
    {
     "cosine": 0.20344148979999432,
     "label": "Good",
     "text": "quick better than a few months ago"
    },
    {
     "cosine": 0.19452847342553825,
     "label": "Bad",
     "text": "very bad coverage wouldnt reach upstairs wouldnt help me out just left after months price went up to from was ment to get mb and only got very bad"
    },
    {
     "cosine": 0.16598428091706804,
     "label": "Bad",
     "text": "wanted to cancel couldnt get through to anyone on the phone just left listening to loud music maybe i would have renewed with them as the service was fine they "
    }
   ]
  },
  {
   "review": "the speed is good but nobody answers when you call and the bill was wrong once",
   "prediction": "Good",
   "probability_good": 0.8004714691705127,
   "decision": "automatic",
   "similar": [
    {
     "cosine": 0.23027440633118365,
     "label": "Bad",
     "text": "internet speed is good but expensive so should be"
    },
    {
     "cosine": 0.21843326865849633,
     "label": "Good",
     "text": "lately in the last months i had no problem at all and the speed is good"
    },
    {
     "cosine": 0.15895407858571348,
     "label": "Bad",
     "text": "i am a home worker and i am paying for m and the speed is speed is mbps this is very has now lowered my expectations"
    }
   ]
  },
  {
   "review": "great deal at first however the router needed replacing twice and they sorted it eventually",
   "prediction": "Good",
   "probability_good": 0.9815926149275948,
   "decision": "automatic",
   "similar": [
    {
     "cosine": 0.20386768538314048,
     "label": "Good",
     "text": "very helpful great deal"
    },
    {
     "cosine": 0.17123080126526347,
     "label": "Good",
     "text": "the service was great and i got a great deal"
    },
    {
     "cosine": 0.1371729045181405,
     "label": "Good",
     "text": "renewed with virgin media and got a great deal"
    }
   ]
  }
 ],
 "digest": {
  "share_good": 0.1375,
  "summary": [
   "im not happy with virgin media",
   "i had phone call to virgin mobile customer services just to change my but i had silly conversation and no have been virgin broadband around more than years and last year i decided to change my mobile to virgin and now i am realy p off with your",
   "i want my contract cancelled"
  ]
 },
 "answers": [
  {
   "question": "What was the cancellation fee for the customer who moved to a neighborhood not serviced by Comcast?",
   "answer": "$756.82",
   "status": "answered",
   "score": 11.859116315841675,
   "doc_id": 2562,
   "evidence": "Now the sticky part it seems that Comcast is dictating where I can live if you have an existing Comcast service contract. If you happen to move to a neighborhood that is not serviced by Comcast you have to pay an outrageous cancellation fee. Mine was $756.82.",
   "type": "MONEY"
  },
  {
   "question": "Which insurance company refused to pay the full electrician estimate after faulty cable wiring?",
   "answer": "Liberty Mutual",
   "status": "answered",
   "score": 10.577487627665201,
   "doc_id": 4307,
   "evidence": "I've gotten multiple estimates from electricians. But Comcast's insurance company (Liberty Mutual) will not pay the full amount, leaving me to pay the difference. I'm being punished for choosing Comcast as my provider.",
   "type": "ENTITY"
  },
  {
   "question": "How much does Comcast
```

Unit tests:

```
..............                                                           [100%]
14 passed in 13.72s

```
